In [19]:
import numpy as np
import pandas as pd

TICK_HZ = 10_000_000  # TIMETICK(f=10000000)

# --- unchanged ---
def build_shared_anchor(*dfs,
                        tick_col="TIMETICK(f=10000000)",
                        group_cols=("Video Name (folder)", "User ID (file)")) -> pd.DataFrame:
    refs = []
    for d in dfs:
        tmp = d[list(group_cols) + [tick_col]].copy()
        tmp[tick_col] = pd.to_numeric(tmp[tick_col], errors="coerce")
        refs.append(tmp.groupby(list(group_cols), as_index=False)[tick_col].min()
                        .rename(columns={tick_col: "min_tick"}))
    ref = (pd.concat(refs, ignore_index=True)
             .groupby(list(group_cols), as_index=False)["min_tick"].min()
             .rename(columns={"min_tick": "anchor_tick"}))
    return ref

def _ensure_labels_upper(df):
    out = df.copy()
    if "label" in out:
        out["label"] = out["label"].astype(str).str.strip().str.upper()
    return out

# --- NEW: sliding window IDs with stride (overlap if stride_sec < win_sec) ---
def add_window_id_sliding(
    df: pd.DataFrame,
    *,
    win_sec: float = 3.0,
    stride_sec: float | None = None,
    tick_col: str = "TIMETICK(f=10000000)",
    group_cols=("Video Name (folder)", "User ID (file)"),
    shared_ref: pd.DataFrame | None = None,
    out_col: str = "win_id",
    out_start_col: str = "win_start_tick",
) -> pd.DataFrame:
    """
    Assign each row to a sliding window that starts every 'stride_sec' seconds,
    each window spanning 'win_sec' seconds. Each sample belongs to exactly one window.
    """
    stride_sec = float(stride_sec if stride_sec is not None else win_sec)
    win_ticks    = int(round(win_sec    * TICK_HZ))
    stride_ticks = int(round(stride_sec * TICK_HZ))

    out = df.copy()
    out[tick_col] = pd.to_numeric(out[tick_col], errors="coerce")

    if shared_ref is not None:
        out = out.merge(shared_ref, on=list(group_cols), how="left", validate="many_to_one")
        anchor = out["anchor_tick"].astype("Int64")
    else:
        anchor = out.groupby(list(group_cols))[tick_col].transform("min").astype("Int64")

    # local time from anchor
    local = (out[tick_col] - anchor).astype("Int64")

    # window index by stride
    k = (local // stride_ticks).astype("Int64")
    start = (k * stride_ticks).astype("Int64")

    # keep only rows that actually fall within [start, start+win)
    in_window = (local - start) < win_ticks
    out = out.loc[in_window].copy()

    out[out_col] = k
    out[out_start_col] = start
    return out

import numpy as np
import pandas as pd

# ---- small robust helpers ----
def trimmed_mean(x, prop=0.1):
    x = pd.Series(x).dropna().to_numpy()
    if x.size == 0:
        return np.nan
    k = int(np.floor(prop * x.size))
    if k == 0:
        return float(np.mean(x))
    xs = np.sort(x)
    return float(np.mean(xs[k: x.size - k]))

def robust_sigma_from_iqr(x):
    """IQR-based std estimate: IQR / 1.349 (≈ std for normal)"""
    x = pd.Series(x).dropna().to_numpy()
    if x.size < 3:
        return np.nan
    q75, q25 = np.percentile(x, [75, 25])
    iqr = q75 - q25
    return float(iqr / 1.349) if iqr > 0 else 0.0

def robust_var_from_mad(x):
    """MAD-based variance estimate (MAD*1.4826)^2"""
    x = pd.Series(x).dropna().to_numpy()
    if x.size < 3:
        return np.nan
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    std = 1.4826 * mad
    return float(std**2)

def circular_variance_deg(angles_deg):
    """Return circular variance in [0,1] for degrees; NaN if <3 samples."""
    a = pd.Series(angles_deg).dropna().to_numpy()
    if a.size < 3:
        return np.nan
    theta = np.deg2rad(a)
    C = np.mean(np.cos(theta))
    S = np.mean(np.sin(theta))
    R = np.sqrt(C**2 + S**2)
    return float(1.0 - R)

# ---- knobs (adjust if needed) ----
MIN_VALID_FRAC = 0.50   # require at least 50% valid gaze samples for rate-like feats
MIN_SAMPLES     = 10    # minimum samples to compute robust stats
EPS             = 1e-9

# 2) FIXATION features (robust)
def window_features_fixations(fix_df: pd.DataFrame, *, win_sec: float = 3.0) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]

    def _agg(w: pd.DataFrame) -> pd.Series:
        # guard
        n = len(w)
        if n < MIN_SAMPLES or "FPOGID" not in w:
            return pd.Series({"fix_efficiency": np.nan, "fix_aoi_ratio": np.nan})

        # valid time proxy: use valid gaze if present, else all rows
        gv = (w["FPOGV"] == 1) if "FPOGV" in w else pd.Series(True, index=w.index)
        valid_frac = float(gv.mean()) if n else 0.0
        valid_time = max(valid_frac * win_sec, EPS)

        # fixation ids + durations
        # (if FPOGD is missing, we just count samples)
        if "FPOGD" in w:
            # aggregate per fixation id to get durations & AOI
            g = w.groupby("FPOGID", dropna=True)
            dur_per_fix = g["FPOGD"].mean()  # more stable than sum on noisy streams
            aoi_per_fix = g["AOI"].apply(lambda s: s.notna().mean()) if "AOI" in w else None
            fix_count = dur_per_fix.size
            # rate per valid second (NOT mean_dur-based)
            fix_efficiency = fix_count / valid_time if valid_frac >= MIN_VALID_FRAC else np.nan
            # time-weighted AOI ratio (duration-weighted)
            if aoi_per_fix is not None and dur_per_fix.notna().any():
                w_aoi = (aoi_per_fix.fillna(0.0) * dur_per_fix.fillna(0.0)).sum()
                fix_aoi_ratio = float(w_aoi / (dur_per_fix.fillna(0.0).sum() + EPS))
            else:
                fix_aoi_ratio = np.nan
        else:
            # fallback: count unique ids or rows
            fix_count = w["FPOGID"].nunique()
            fix_efficiency = fix_count / valid_time if valid_frac >= MIN_VALID_FRAC else np.nan
            fix_aoi_ratio = w["AOI"].notna().mean() if "AOI" in w else np.nan

        return pd.Series({
            "fix_efficiency": fix_efficiency,
            "fix_aoi_ratio": fix_aoi_ratio,
        })

    return (fix_df.dropna(subset=["win_id"])
                 .groupby(keys, dropna=False)
                 .apply(_agg)
                 .reset_index())

# 3) GAZE/PUPIL/SACCADE/BLINK features (robust)
def window_features_gaze(gaze_df: pd.DataFrame, *, win_sec: float = 3.0) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]

    def _agg(w: pd.DataFrame) -> pd.Series:
        n = len(w)
        if n == 0:
            return pd.Series({
                "gaze_dispersion_total": np.nan,
                "pupil_mean": np.nan, "pupil_var": np.nan,
                "saccade_mean_mag": np.nan, "saccade_dir_var": np.nan,
                "blink_rate": np.nan, "valid_gaze_ratio": np.nan
            })

        gv = (w["FPOGV"] == 1) if "FPOGV" in w else pd.Series(True, index=w.index)
        valid_frac = float(gv.mean()) if n else 0.0
        valid_time = max(valid_frac * win_sec, EPS)

        # --- Gaze dispersion: robust std via IQR on valid samples
        gx = w.loc[gv, "FPOGX"] if "FPOGX" in w else pd.Series(dtype=float)
        gy = w.loc[gv, "FPOGY"] if "FPOGY" in w else pd.Series(dtype=float)
        sx = robust_sigma_from_iqr(gx)
        sy = robust_sigma_from_iqr(gy)
        gaze_disp_total = np.sqrt((0 if pd.isna(sx) else sx**2) + (0 if pd.isna(sy) else sy**2)) \
                          if (gx.size >= MIN_SAMPLES and gy.size >= MIN_SAMPLES) else np.nan

        # --- Pupil: rowwise mean across valid eyes; robust var via MAD
        if "LPMM" in w and "RPMM" in w:
            lv = (w["LPMMV"] == 1) if "LPMMV" in w else pd.Series(False, index=w.index)
            rv = (w["RPMMV"] == 1) if "RPMMV" in w else pd.Series(False, index=w.index)
            l = w["LPMM"].where(lv)
            r = w["RPMM"].where(rv)
            pupil_row = pd.concat([l, r], axis=1).mean(axis=1, skipna=True)
            pupil_vals = pupil_row[gv].dropna()
            pupil_mean = float(pupil_vals.median()) if pupil_vals.size >= MIN_SAMPLES else np.nan
            pupil_var  = robust_var_from_mad(pupil_vals) if pupil_vals.size >= MIN_SAMPLES else np.nan
        else:
            pupil_mean, pupil_var = np.nan, np.nan

        # --- Saccades: trimmed mean magnitude; circular variance for direction
        if "SACCADE_MAG" in w and gv.any():
            mags = w.loc[gv, "SACCADE_MAG"].dropna()
            saccade_mean_mag = float(trimmed_mean(mags, 0.1)) if mags.size >= MIN_SAMPLES else np.nan
        else:
            saccade_mean_mag = np.nan

        if "SACCADE_DIR" in w and gv.any():
            sdir = w.loc[gv, "SACCADE_DIR"].dropna()
            saccade_dir_var = circular_variance_deg(sdir) if sdir.size >= MIN_SAMPLES else np.nan
        else:
            saccade_dir_var = np.nan

        # --- Blinks: per valid second
        if "BKID" in w:
            blink_count = w.loc[gv, "BKID"].dropna().nunique()
            blink_rate  = blink_count / valid_time if valid_frac >= MIN_VALID_FRAC else np.nan
        else:
            blink_rate = np.nan

        return pd.Series({
            "gaze_dispersion_total": gaze_disp_total,
            "pupil_mean": pupil_mean,
            "pupil_var": 0.0 if pd.isna(pupil_var) else float(pupil_var),  # keep your zero-fill convention
            "saccade_mean_mag": saccade_mean_mag,
            "saccade_dir_var": 0.0 if pd.isna(saccade_dir_var) else float(saccade_dir_var),
            "blink_rate": blink_rate,
            "valid_gaze_ratio": valid_frac if n else np.nan,
        })

    return (gaze_df.dropna(subset=["win_id"])
                 .groupby(keys, dropna=False)
                 .apply(_agg)
                 .reset_index())

def combine_fix_gaze_windows(fix_w: pd.DataFrame, gaze_w: pd.DataFrame) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]
    merged = pd.merge(fix_w, gaze_w, on=keys, how="outer", validate="one_to_one", suffixes=("_fix","_gaze"))
    # resolve on_fraction: prefer gaze (usually denser), else fix
    if "on_fraction_gaze" in merged.columns or "on_fraction_fix" in merged.columns:
        merged["on_fraction"] = merged.get("on_fraction_gaze", np.nan)
        merged["on_fraction"] = merged["on_fraction"].fillna(merged.get("on_fraction_fix", np.nan))
    return merged

In [22]:
def make_family_windows_with_overlap(
    fix_df: pd.DataFrame,
    gaze_df: pd.DataFrame,
    *,
    fam: int,
    win_sec: float = 3.0,
    stride_sec: float | None = None,
    qc_min_valid: float = 0.5,
    on_thresh_on: float = 0.80,
    on_thresh_off: float = 0.20,
    save_path: str | None = None,
) -> pd.DataFrame:
    """
    Build windows with overlap, compute features + on_fraction,
    and relabel windows with purity thresholds (drop ambiguous).
    """
    if fix_df.empty and gaze_df.empty:
        print(f"[family {fam}] Empty split; skipping.")
        return pd.DataFrame()

    fix_df = _ensure_labels_upper(fix_df)
    gaze_df = _ensure_labels_upper(gaze_df)

    anchor = build_shared_anchor(fix_df, gaze_df)

    fix_wid  = add_window_id_sliding(fix_df,  win_sec=win_sec, stride_sec=stride_sec, shared_ref=anchor)
    gaze_wid = add_window_id_sliding(gaze_df, win_sec=win_sec, stride_sec=stride_sec, shared_ref=anchor)

    fix_w = window_features_fixations(fix_wid)
    gaz_w = window_features_gaze(gaze_wid, win_sec=win_sec)

    win = combine_fix_gaze_windows(fix_w, gaz_w)

    # quality + purity filter
    if "valid_gaze_ratio" in win:
        win = win.loc[win["valid_gaze_ratio"] >= qc_min_valid].copy()

    # decide final window label from purity
    def _purity_label(frac):
        if pd.isna(frac): return np.nan
        if frac >= on_thresh_on:  return "ON"
        if frac <= on_thresh_off: return "OFF"
        return np.nan  # ambiguous

    win["label"] = win["on_fraction"].map(_purity_label)
    kept = win["label"].notna().sum()
    print(f"[family {fam}] kept {kept}/{len(win)} windows after purity thresholds "
          f"(on_fraction ≥{on_thresh_on} → ON, ≤{on_thresh_off} → OFF)")

    # reorder columns for convenience
    cols = ["Video Name (folder)","User ID (file)","Task Id","type","phase","block_id","win_id",
            "label","on_fraction","valid_gaze_ratio",
            "fix_efficiency","fix_aoi_ratio",
            "gaze_dispersion_total","pupil_mean","pupil_var",
            "saccade_mean_mag","saccade_dir_var","blink_rate"]
    cols = [c for c in cols if c in win.columns]
    win = win.loc[win["label"].notna(), cols].reset_index(drop=True)

    if save_path is not None:
        win.to_csv(save_path, index=False)
        print(f"[family {fam}] saved → {save_path} (rows={len(win)})")

    return win

In [20]:
fix_df1 = pd.read_csv("outputs/fix_onoff_task1.csv")
fix_df2 = pd.read_csv("outputs/fix_onoff_task2.csv")
fix_df3 = pd.read_csv("outputs/fix_onoff_task3.csv")

gaz_df1 = pd.read_csv("outputs/gaze_onoff_task1.csv")
gaz_df2 = pd.read_csv("outputs/gaze_onoff_task2.csv")
gaz_df3 = pd.read_csv("outputs/gaze_onoff_task3.csv")

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_26114/367853894.py:6: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  gaz_df2 = pd.read_csv("outputs/gaze_onoff_task2.csv")


In [13]:

fix_fams = {1: fix_df1, 2: fix_df2, 3: fix_df3}
gaz_fams = {1: gaz_df1, 2: gaz_df2, 3: gaz_df3}

In [ ]:
win_grid   = [4.0]     # seconds
stride_ops = ["full"]              # stride = win or win/2
QC_MIN_VALID   = 0.5
TH_ON, TH_OFF  = 0.80, 0.20

all_ready = {}  # (fam, win, stride_tag) -> windowed df

for fam in (1, 2, 3):
    for w in win_grid:
        for s_tag in stride_ops:
            stride = w if s_tag == "full" else (w/2.0)
            out_path = f"outputs/windows_task{fam}_w{w:g}_s{s_tag}.csv"
            dfw = make_family_windows_with_overlap(
                fix_fams[fam], gaz_fams[fam],
                fam=fam,
                win_sec=w,
                stride_sec=stride,
                qc_min_valid=QC_MIN_VALID,
                on_thresh_on=TH_ON,
                on_thresh_off=TH_OFF,
                save_path=out_path,
            )
            all_ready[(fam, w, s_tag)] = dfw

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_26114/3616127955.py:164: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_26114/3616127955.py:238: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)


KeyError: 'on_fraction'